In [1]:
# Load libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.model_selection import cross_val_score

In [2]:
# Load data
forest_fires = pd.read_csv("data/forest_fires_train.csv")

# Create target variable y (big_fire)
y = forest_fires['big_fire']

# Drop the target variable from our predictor dataset
drop_list = ['big_fire']
x_data = forest_fires.drop(columns=drop_list)
X = x_data.to_numpy()


# This splits the forest fires dataset into 80% training, 10% validation, and 10% testing sets

# Split data into test, train, validation splits
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=1
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=1
)

# Scale our train, test and validation data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Print some information
print('X dim:', X.shape, 'y dim:', y.shape)
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

X dim: (24637, 15) y dim: (24637,)
Training set size: 19709
Validation set size: 2464
Test set size: 2464


In [ ]:
iterations = 200000

# Create the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=iterations, random_state=42))
])

# Create parameter grid
param_grid = {
    'mlp__hidden_layer_sizes': [(50,), (50, 50), (50, 50, 25)],
    'mlp__activation': ['tanh', 'relu'],
    'mlp__solver': ['adam'],
    'mlp__alpha': [0.0001, 0.001, 0.01, 0.05, 0.1],
    'mlp__learning_rate': ['constant', 'adaptive'],
}

grid = GridSearchCV(pipeline, param_grid, n_jobs=-1, cv=5, scoring='f1')

grid.fit(X_train, y_train)

best_params = grid.best_params_
print(f"Best Parameters: {best_params}")
print(f"Best Score: {grid.best_score_}")


Best Parameters: {'mlp__activation': 'tanh', 'mlp__alpha': 0.05, 'mlp__hidden_layer_sizes': (50, 50), 'mlp__learning_rate': 'constant', 'mlp__solver': 'adam'}
Best Score: 0.546692890149267


In [ ]:
iterations = 200000

# Create the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=iterations, random_state=42))
])

# Create parameter grid
param_grid = {
    'mlp__hidden_layer_sizes': [(50, 50), (50, 100), (25, 100), (100, 25), (100, 50)],
    'mlp__activation': ['tanh'],
    'mlp__solver': ['adam'],
    'mlp__alpha': [0.05],
    'mlp__learning_rate': ['constant'],
}

grid2 = GridSearchCV(pipeline, param_grid, n_jobs=-1, cv=5, scoring='f1')

grid2.fit(X_train, y_train)

best_params = grid2.best_params_
print(f"Best Parameters: {best_params}")
print(f"Best Score: {grid2.best_score_}")

In [ ]:
iterations = 200000

# Create the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=iterations, random_state=42))
])

# Create parameter grid
param_grid = {
    'mlp__hidden_layer_sizes': [(30, 30), (40, 40), (50, 50), (60, 60), (70, 70)],
    'mlp__activation': ['tanh'],
    'mlp__solver': ['adam'],
    'mlp__alpha': [0.05],
    'mlp__learning_rate': ['constant'],
}

grid3 = GridSearchCV(pipeline, param_grid, n_jobs=-1, cv=5, scoring='f1')

grid3.fit(X_train, y_train)

best_params = grid3.best_params_
print(f"Best Parameters: {best_params}")
print(f"Best Score: {grid3.best_score_}")

Best Parameters: {'mlp__activation': 'tanh', 'mlp__alpha': 0.05, 'mlp__hidden_layer_sizes': (50, 50), 'mlp__learning_rate': 'constant', 'mlp__solver': 'adam'}
Best Score: 0.546692890149267


In [9]:
# MLPClassifier with X_train_scaled
mlp = MLPClassifier(hidden_layer_sizes=(50, 50), 
                    alpha=0.05,
                    activation='tanh',
                    learning_rate='adaptive',
                    solver='adam',
                    max_iter=200000)

mlp.fit(X_train_scaled, y_train)
y_pred = mlp.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

print('F1 Score:', f1_score(y_test, y_pred))
print('AUC:', roc_auc_score(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.91      0.96      0.93      2092
           1       0.67      0.49      0.57       372

    accuracy                           0.89      2464
   macro avg       0.79      0.72      0.75      2464
weighted avg       0.88      0.89      0.88      2464

F1 Score: 0.56656346749226
AUC: 0.7242182199469561


In [18]:
# MLPClassifier with SMOTE applied to X_train_scaled
sm = SMOTE(random_state=1)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

mlp = MLPClassifier(hidden_layer_sizes=(50, 50), 
                    alpha=0.05,
                    activation='tanh',
                    learning_rate='adaptive',
                    solver='adam',
                    max_iter=200000)

mlp.fit(X_train_res, y_train_res)
y_pred = mlp.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

print('F1 Score:', f1_score(y_test, y_pred))
print('AUC:', roc_auc_score(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.94      0.84      0.89      2092
           1       0.43      0.69      0.53       372

    accuracy                           0.82      2464
   macro avg       0.68      0.76      0.71      2464
weighted avg       0.86      0.82      0.83      2464

F1 Score: 0.5300207039337475
AUC: 0.7633020826908449


In [20]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sm = SMOTE(random_state=1)
X_res, y_res = sm.fit_resample(X_dev, y_dev)

# Define the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(50, 50), 
                    alpha=0.05,
                    activation='tanh',
                    learning_rate='adaptive',
                    solver='adam',
                    max_iter=200000))
])

# Use StratifiedKFold for classification to maintain class proportions
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Execute cross-validation
f1_scores_val = cross_val_score(pipeline, X_res, y_res, cv=cv, scoring='f1')
auc_scores_val = cross_val_score(pipeline, X_res, y_res, cv=cv, scoring='roc_auc')

pipeline.fit(X_res, y_res)
y_pred = pipeline.predict(X_test)
final_score = pipeline.score(X_test, y_test)

print('F1 Validation', f1_scores_val.mean())
print('AUC Validation', auc_scores_val.mean())

print('F1 Test:', f1_score(y_test, y_pred))
print('AUC Test:', roc_auc_score(y_test, y_pred))

F1 Validation 0.8566469112348651
AUC Validation 0.9253071898746434
F1 Test: 0.5477903391572456
AUC Test: 0.77018417285323
